# Magana-ATLAS 8B LoRA (Powered by Awarri)

QLoRA fine-tune of N-ATLaS so it explains schoolwork **without writing numbers**: it copies Magana's ⟦N⟧ markers and refuses calculations. Then the 40-page benchmark is run on base N-ATLaS and the adapter in the same session.

**Kaggle settings:** Accelerator = GPU T4, Internet = On, Add-ons → Secrets → attach `HF_TOKEN` (a Hugging Face token from an account that accepted the N-ATLaS terms).

Only team-written pages are used (Magana docs/MODELS.md: no learners' papers on Kaggle). Recipe: Ibà-ATLAS (github.com/jemiridaniel/iba-natlas).

In [ ]:
!pip install -q -U "transformers>=4.46" peft bitsandbytes accelerate httpx pydantic

In [ ]:
# Cloned outside /kaggle/working, so the outputs are just the two zips.
%cd /tmp
!git clone -q https://github.com/jemiridaniel/magana.git
%cd magana
!git log -1 --oneline

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
try:
    login(UserSecretsClient().get_secret("HF_TOKEN"))
except Exception as e:
    raise SystemExit("Attach the HF_TOKEN secret to THIS notebook (Add-ons → Secrets → HF_TOKEN), then run again: " + repr(e))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Build the training set (English only until the drafts are reviewed)

In [ ]:
!python data/lora/build.py | head -20

## 2. QLoRA fine-tune (≈15 min on a T4, plus the 16 GB download)

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python notebooks/magana-lora/train.py --epochs 3 --out outputs/Magana-ATLAS-8B-LoRA-Powered-by-Awarri

## 3. The 40-page benchmark: base and adapter, same session (20 schoolwork prompts each)

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python notebooks/magana-lora/bench.py --adapter outputs/Magana-ATLAS-8B-LoRA-Powered-by-Awarri --out results/lora-bench.jsonl

In [ ]:
import json, collections
rows = [json.loads(l) for l in open('results/lora-bench.jsonl')]
for m in ('base', 'magana-lora'):
    mine = [r for r in rows if r['model'] == m]
    print(m, 'passed', sum(not r['blocked'] for r in mine), 'of', len(mine),
          collections.Counter(p['code'] for r in mine for p in r['problems']))

## 4. Save the adapter (with its licence and attribution) and the results

In [ ]:
!cp notebooks/magana-lora/MODEL_CARD.md outputs/Magana-ATLAS-8B-LoRA-Powered-by-Awarri/README.md
!cp outputs/Magana-ATLAS-8B-LoRA-Powered-by-Awarri/train_log.json results/
!cp data/lora/dataset_card.json results/
!cd outputs && zip -qr /kaggle/working/Magana-ATLAS-8B-LoRA-Powered-by-Awarri.zip Magana-ATLAS-8B-LoRA-Powered-by-Awarri
!zip -qr /kaggle/working/magana_lora_results.zip results
print('Outputs: the adapter zip and magana_lora_results.zip')